In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glasbey

def umap_to_plotly(
    adata,
    color,
    basis="X_umap",
    remove_frame=False,
    palette=None,       # list of hex colors, or "glasbey" to auto-generate
    colormap=None,      # matplotlib colormap name for continuous data, e.g. "viridis"
    fig=None,           # existing go.Figure (from make_subplots) to add traces to
    row=None,           # subplot row (1-indexed)
    col=None,           # subplot col (1-indexed)
    title=None,
    **kwargs,
):
    embedding = adata.obsm[basis]
    ax1, ax2 = basis.lstrip("X_").upper(), basis.lstrip("X_").upper()
    dim_name = basis.lstrip("X_")
    coord_cols = [f"{dim_name.upper()}1", f"{dim_name.upper()}2"]

    df = adata.obs[[color]].copy() if color in adata.obs else adata.obs.copy()
    df[coord_cols[0]] = embedding[:, 0]
    df[coord_cols[1]] = embedding[:, 1]
    if color not in df:
        df[color] = adata.obs[color].values

    is_categorical = pd.api.types.is_categorical_dtype(df[color]) or pd.api.types.is_object_dtype(df[color])

    subplot_kwargs = dict(row=row, col=col) if (row and col) else {}
    standalone = fig is None
    if standalone:
        fig = go.Figure()

    if is_categorical:
        categories = df[color].astype("category").cat.categories.tolist()

        if palette == "glasbey" or (palette is None):
            colors = glasbey.create_palette(palette_size=len(categories))
        else:
            colors = palette

        color_map = dict(zip(categories, colors))

        for cat in categories:
            mask = df[color].astype(str) == str(cat)
            fig.add_trace(
                go.Scatter(
                    x=df.loc[mask, coord_cols[0]],
                    y=df.loc[mask, coord_cols[1]],
                    mode="markers",
                    name=str(cat),
                    marker=dict(color=color_map[str(cat)], size=4, opacity=0.8),
                    text=df.loc[mask, color].astype(str),
                    hovertemplate=f"{color}: %{{text}}<br>{coord_cols[0]}: %{{x:.2f}}<br>{coord_cols[1]}: %{{y:.2f}}<extra></extra>",
                    **kwargs,
                ),
                **subplot_kwargs,
            )
    else:
        cmap_name = colormap or "viridis"
        cmap = plt.get_cmap(cmap_name)
        vals = df[color].values.astype(float)
        norm = (vals - vals.min()) / (vals.ptp() or 1)
        hex_colors = [f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}" for r, g, b, _ in cmap(norm)]

        fig.add_trace(
            go.Scatter(
                x=df[coord_cols[0]],
                y=df[coord_cols[1]],
                mode="markers",
                marker=dict(
                    color=vals,
                    colorscale=cmap_name,
                    showscale=True,
                    size=4,
                    opacity=0.8,
                    colorbar=dict(title=color),
                ),
                text=df[color].astype(str),
                hovertemplate=f"{color}: %{{text}}<br>{coord_cols[0]}: %{{x:.2f}}<br>{coord_cols[1]}: %{{y:.2f}}<extra></extra>",
                **kwargs,
            ),
            **subplot_kwargs,
        )

    axis_style = dict(
        showline=not remove_frame,
        showticklabels=not remove_frame,
        showgrid=False,
        zeroline=False,
        ticks="" if remove_frame else "outside",
    )

    if standalone:
        fig.update_layout(
            title=title,
            xaxis=dict(title=coord_cols[0], **axis_style),
            yaxis=dict(title=coord_cols[1], **axis_style, scaleanchor="x"),
            plot_bgcolor="white",
            legend=dict(itemsizing="constant"),
        )
    else:
        # Update the specific subplot axes
        axis_idx = "" if (row == 1 and col == 1) else (col + (row - 1) * col)
        fig.update_xaxes(title_text=coord_cols[0], **axis_style, row=row, col=col)
        fig.update_yaxes(title_text=coord_cols[1], **axis_style, row=row, col=col)
        if title:
            fig.layout.annotations[((row - 1) * col + col) - 1].text = title

    return fig